# Qwen Structured Extraction — Experiment v2

Test 3 models for extracting structured CNIE fields from raw PaddleOCR output:

| # | Model | RAM | Why test it |
|---|---|---|---|
| A | **Qwen3-8B Q4_K_M** | ~5.5GB | Newest, smartest, best shot |
| B | **Qwen2.5-7B Q4_K_M** | ~5.5GB | Proven for structured extraction |
| C | **Qwen3-4B Q5_K_M** | ~3GB | Fallback if 7-8B too slow |

**Goal:** OCR text (Arabic + French) -> clean structured JSON on CPU

**How to use:** Run Step 1, restart runtime, then run all other cells in order.
To switch model: change the uncommented line in Step 3, restart runtime, re-run.

In [ ]:
# Step 1: Install (run once, then restart runtime)
!pip install llama-cpp-python huggingface-hub

print("\n" + "="*50)
print("DONE! Now go to Runtime > Restart session")
print("Then skip this cell and run the next ones.")
print("="*50)

In [ ]:
# Step 2: Verify
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
import time, json, os
print("All imports OK!")

---
## Step 3: Choose & Download Model

Uncomment ONE model block. Start with **Model A (Qwen3-8B)** first.

In [ ]:
# =============================================
# MODEL A: Qwen3-8B Q4_K_M (~5GB) — TRY FIRST
# Newest architecture, best quality, 201 languages
# =============================================
MODEL_REPO = "Qwen/Qwen3-8B-GGUF"
MODEL_FILE = "Qwen3-8B-Q4_K_M.gguf"
MODEL_LABEL = "Qwen3-8B Q4_K_M"

# =============================================
# MODEL B: Qwen2.5-7B Q4_K_M (~4.7GB)
# Proven, stable, great for structured output
# (using bartowski repo — single file, no splits)
# =============================================
# MODEL_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
# MODEL_FILE = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
# MODEL_LABEL = "Qwen2.5-7B Q4_K_M"

# =============================================
# MODEL C: Qwen3-4B Q5_K_M (~2.9GB)
# Lighter fallback if 7-8B too slow on your CPU
# =============================================
# MODEL_REPO = "Qwen/Qwen3-4B-GGUF"
# MODEL_FILE = "Qwen3-4B-Q5_K_M.gguf"
# MODEL_LABEL = "Qwen3-4B Q5_K_M"

print(f"Downloading {MODEL_LABEL}...")
print("(This may take a few minutes on first run)")
t0 = time.time()
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(f"Downloaded in {time.time()-t0:.1f}s")
print(f"Path: {model_path}")

In [ ]:
# Step 4: Load model
n_threads = os.cpu_count() or 4
print(f"Loading {MODEL_LABEL} with {n_threads} threads...")

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=n_threads,
    verbose=False,
)
print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Ready to test: {MODEL_LABEL}")

---
## Step 5: System Prompt + Extraction Function

In [ ]:
import re

SYSTEM_PROMPT = """You are a Moroccan ID card (CNIE) data extraction assistant.

You receive raw OCR text from a CNIE card in Arabic and French.
Extract the fields and return ONLY valid JSON, nothing else.

Fields to extract:
- last_name_fr: Family name in French (Latin letters)
- first_name_fr: First name in French (Latin letters)
- last_name_ar: Family name in Arabic
- first_name_ar: First name in Arabic
- birth_date: Date of birth (DD.MM.YYYY format)
- birth_place_fr: Place of birth in French
- birth_place_ar: Place of birth in Arabic
- card_number: CIN number (e.g. AB123456)
- expiry_date: Card expiry date (DD.MM.YYYY format)
- gender: M or F

Rules:
- If a field is not found, set it to null
- Fix obvious OCR errors (e.g. 0 instead of O in names, 1 instead of I)
- Return ONLY the JSON object, no explanation
- Do NOT include any thinking or reasoning, just the JSON
"""


def parse_json_from_response(raw):
    """Extract JSON from model response, handling thinking tags and extra text."""
    # Remove <think>...</think> blocks if present (Qwen3 thinking mode)
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()

    # Try direct parse first
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Try to find JSON object in the text
    match = re.search(r'\{[^{}]*\}', cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    # Try multiline/nested JSON
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    return {"_raw": raw, "_error": "json_parse_failed"}


# Detect if model is Qwen3 (thinking model) — affects how we call it
IS_QWEN3 = "Qwen3" in MODEL_LABEL
print(f"Model: {MODEL_LABEL}")
print(f"Thinking model (Qwen3): {IS_QWEN3}")
if IS_QWEN3:
    print("-> Will disable thinking mode and skip response_format constraint")


def extract_fields(ocr_text, system_prompt=SYSTEM_PROMPT, verbose=True):
    """Send OCR text to the model and get structured JSON back."""
    t0 = time.time()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract fields from this OCR output:\n\n{ocr_text}"},
    ]

    # Qwen3 thinking models: don't use response_format (it conflicts with <think> tags)
    # Instead, we parse JSON from the raw output
    if IS_QWEN3:
        # Add /no_think to disable thinking and get direct JSON
        messages[1]["content"] += "\n\n/no_think"
        response = llm.create_chat_completion(
            messages=messages,
            max_tokens=512,
            temperature=0.1,
        )
    else:
        # Qwen2.5 and other non-thinking models: use JSON mode
        response = llm.create_chat_completion(
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=512,
            temperature=0.1,
        )

    elapsed = time.time() - t0
    raw = response["choices"][0]["message"]["content"]
    usage = response.get("usage", {})

    if verbose:
        print(f"Model: {MODEL_LABEL}")
        print(f"Inference time: {elapsed:.2f}s")
        print(f"Tokens — prompt: {usage.get('prompt_tokens', '?')}, "
              f"completion: {usage.get('completion_tokens', '?')}")
        if IS_QWEN3:
            print(f"Raw output (first 500 chars):\n{raw[:500]}")

    parsed = parse_json_from_response(raw)

    return {"fields": parsed, "time_s": round(elapsed, 2), "model": MODEL_LABEL, "raw": raw}


print("System prompt + extract_fields() ready.")

---
## Step 6: Test Samples

3 test cases: clean OCR, noisy OCR, and your real data.

In [ ]:
# ============================================================
# SAMPLE 1: Clean OCR
# ============================================================
SAMPLE_OCR_1 = """
Arabic OCR:
  - text: 'المملكة المغربية', confidence: 0.95
  - text: 'بطاقة التعريف الوطنية', confidence: 0.92
  - text: 'الشافعي', confidence: 0.88
  - text: 'بلال', confidence: 0.91
  - text: 'المزداد بتاريخ', confidence: 0.85
  - text: 'بالرباط', confidence: 0.90

French OCR:
  - text: 'ROYAUME DU MAROC', confidence: 0.97
  - text: 'CHAFI', confidence: 0.93
  - text: 'BILAL', confidence: 0.95
  - text: 'Ne le 22.01.2007', confidence: 0.89
  - text: 'a RABAT', confidence: 0.92
  - text: 'Valable jusqu au 19.03.2029', confidence: 0.90
  - text: 'AB123456', confidence: 0.94
  - text: 'M', confidence: 0.96
"""

# Expected correct output for scoring:
EXPECTED_1 = {
    "last_name_fr": "CHAFI",
    "first_name_fr": "BILAL",
    "last_name_ar": "الشافعي",
    "first_name_ar": "بلال",
    "birth_date": "22.01.2007",
    "birth_place_fr": "RABAT",
    "birth_place_ar": "الرباط",
    "card_number": "AB123456",
    "expiry_date": "19.03.2029",
    "gender": "M",
}

print("=" * 60)
print(f"TEST 1 — Clean OCR — {MODEL_LABEL}")
print("=" * 60)
result1 = extract_fields(SAMPLE_OCR_1)
print("\nExtracted:")
print(json.dumps(result1["fields"], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# SAMPLE 2: Noisy OCR (OCR errors: 0/O, 1/I, 7/T swaps)
# ============================================================
SAMPLE_OCR_2 = """
Arabic OCR:
  - text: 'الملكة المكربية', confidence: 0.72
  - text: 'بطاقة التعريف الوطنيى', confidence: 0.68
  - text: 'لوح', confidence: 0.55
  - text: 'اجا', confidence: 0.60

French OCR:
  - text: 'R0YAUME DU MAR0C', confidence: 0.75
  - text: 'CHAF1', confidence: 0.70
  - text: 'B1LAL', confidence: 0.72
  - text: 'Ne le 22.O1.20O7', confidence: 0.65
  - text: 'a RABA7', confidence: 0.60
  - text: 'Va1ab1e jusqu au 19.03.2029', confidence: 0.58
  - text: 'A8123456', confidence: 0.55
"""

print("=" * 60)
print(f"TEST 2 — Noisy OCR — {MODEL_LABEL}")
print("=" * 60)
result2 = extract_fields(SAMPLE_OCR_2)
print("\nExtracted:")
print(json.dumps(result2["fields"], indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# SAMPLE 3: Paste YOUR real PaddleOCR output here
# ============================================================
REAL_OCR = """
PASTE YOUR REAL OCR OUTPUT HERE
"""

# Uncomment to run:
# print("=" * 60)
# print(f"TEST 3 — Real OCR — {MODEL_LABEL}")
# print("=" * 60)
# result3 = extract_fields(REAL_OCR)
# print("\nExtracted:")
# print(json.dumps(result3["fields"], indent=2, ensure_ascii=False))

---
## Step 7: Auto-Score Results

Automatically compare extracted fields against expected values.

In [ ]:
def score_extraction(extracted, expected):
    """Compare extracted fields to expected. Returns (correct, total, details)."""
    correct = 0
    total = len(expected)
    details = []

    for field, exp_val in expected.items():
        got = extracted.get(field)
        # Normalize for comparison
        exp_norm = str(exp_val).strip().upper() if exp_val else None
        got_norm = str(got).strip().upper() if got else None

        match = exp_norm == got_norm
        if match:
            correct += 1
            details.append(f"  OK  {field}: '{got}'")
        else:
            details.append(f"  MISS {field}: expected '{exp_val}' got '{got}'")

    return correct, total, details


print(f"\nSCORECARD — {MODEL_LABEL} — Sample 1 (clean OCR):")
print("-" * 60)
correct, total, details = score_extraction(result1["fields"], EXPECTED_1)
for d in details:
    print(d)
print(f"\nScore: {correct}/{total} fields correct | Time: {result1['time_s']}s")

---
## Step 8: Few-Shot Prompt (if zero-shot wasn't good enough)

In [ ]:
SYSTEM_PROMPT_FEWSHOT = """You are a Moroccan ID card (CNIE) data extraction assistant.
You receive raw OCR text from a CNIE card in Arabic and French.
Extract the fields and return ONLY valid JSON.

Example input:
Arabic OCR:
  - text: 'العلوي', confidence: 0.90
  - text: 'محمد', confidence: 0.88
French OCR:
  - text: 'ALAOUI', confidence: 0.93
  - text: 'MOHAMMED', confidence: 0.91
  - text: 'Ne le 15.06.1990', confidence: 0.89
  - text: 'a CASABLANCA', confidence: 0.85
  - text: 'Valable jusqu au 20.06.2030', confidence: 0.87
  - text: 'CD987654', confidence: 0.92
  - text: 'M', confidence: 0.95

Example output:
{"last_name_fr": "ALAOUI", "first_name_fr": "MOHAMMED", "last_name_ar": "العلوي", "first_name_ar": "محمد", "birth_date": "15.06.1990", "birth_place_fr": "CASABLANCA", "birth_place_ar": null, "card_number": "CD987654", "expiry_date": "20.06.2030", "gender": "M"}

Rules:
- If a field is not found, set it to null
- Fix obvious OCR errors (e.g. 0 instead of O in names, 1 instead of I)
- Return ONLY the JSON object, no explanation
"""

print("=" * 60)
print(f"FEW-SHOT TEST — {MODEL_LABEL} — Sample 2 (noisy)")
print("=" * 60)
result_fs = extract_fields(SAMPLE_OCR_2, system_prompt=SYSTEM_PROMPT_FEWSHOT)
print("\nFew-shot extracted:")
print(json.dumps(result_fs["fields"], indent=2, ensure_ascii=False))

print("\n" + "=" * 60)
print("Compare with zero-shot on same input:")
print(json.dumps(result2["fields"], indent=2, ensure_ascii=False))

---
## Step 9: Comparison Table

Fill in as you test each model. Switch model in Step 3, restart runtime, re-run all.

In [ ]:
# ============================================================
# FILL IN YOUR RESULTS AFTER TESTING EACH MODEL
# ============================================================

comparison = {
    "Qwen3-8B Q4": {
        "ram_gb": 5.5,
        "time_clean_s": None,     # result1["time_s"]
        "time_noisy_s": None,     # result2["time_s"]
        "score_clean": None,      # e.g. "8/10"
        "score_noisy": None,      # e.g. "6/10"
        "json_valid": None,       # True/False
        "arabic_quality": "",     # good/bad/partial
        "notes": "",
    },
    "Qwen2.5-7B Q4": {
        "ram_gb": 5.5,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_quality": "",
        "notes": "",
    },
    "Qwen3-4B Q5": {
        "ram_gb": 3.0,
        "time_clean_s": None,
        "time_noisy_s": None,
        "score_clean": None,
        "score_noisy": None,
        "json_valid": None,
        "arabic_quality": "",
        "notes": "",
    },
}

# Pretty print
print(f"{'Model':<20} {'RAM':<8} {'Clean(s)':<10} {'Noisy(s)':<10} {'Clean':<8} {'Noisy':<8} {'JSON':<8} {'Arabic':<10} {'Notes'}")
print("=" * 110)
for model, d in comparison.items():
    print(f"{model:<20} {str(d['ram_gb'])+'GB':<8} {str(d['time_clean_s'] or '—'):<10} {str(d['time_noisy_s'] or '—'):<10} "
          f"{str(d['score_clean'] or '—'):<8} {str(d['score_noisy'] or '—'):<8} {str(d['json_valid'] or '—'):<8} "
          f"{d['arabic_quality'] or '—':<10} {d['notes']}")

---
## Step 10: Bridge Function (for Flask integration later)

This is the function we'll reuse when integrating into `app.py`.

In [ ]:
def format_ocr_for_llm(arabic_results, french_results):
    """
    Format PaddleOCR output into the text format the LLM expects.
    This is the bridge between OCR and LLM.
    
    Args:
        arabic_results: list of {"text": ..., "confidence": ...}
        french_results: list of {"text": ..., "confidence": ...}
    Returns:
        Formatted string for the LLM prompt
    """
    lines = ["Arabic OCR:"]
    for d in arabic_results:
        lines.append(f"  - text: '{d['text']}', confidence: {d['confidence']}")
    lines.append("")
    lines.append("French OCR:")
    for d in french_results:
        lines.append(f"  - text: '{d['text']}', confidence: {d['confidence']}")
    return "\n".join(lines)


# Quick test with mock data
mock_ar = [{"text": "الشافعي", "confidence": 0.88}, {"text": "بلال", "confidence": 0.91}]
mock_fr = [{"text": "CHAFI", "confidence": 0.93}, {"text": "BILAL", "confidence": 0.95}]
formatted = format_ocr_for_llm(mock_ar, mock_fr)
print("Formatted OCR for LLM:")
print(formatted)
print("\nThis function will be copied to services/llm_extractor.py")

---
## Decision Checklist

After testing all 3 models, answer:

1. **Winner model?** Which gave best score on both clean and noisy?
2. **Zero-shot or few-shot?** Which prompt style was better?
3. **Speed OK?** Under 5s on Colab CPU? (local will be similar or faster)
4. **Arabic quality?** Did it correctly extract Arabic names?
5. **JSON reliable?** Did it always return valid parseable JSON?

Share your answers and we'll integrate the winner into the Flask pipeline.